# SageMaker Pipeline BYOC — Predicción de Ventas

Pipeline completo que encadena los containers BYOC de preprocessing y training
con un step de evaluación, registro condicional del modelo y batch transform.

## Flujo del pipeline

```
ProcessingStep (preprocess.py)
        ↓
TrainingStep (container BYOC training)
        ↓
ProcessingStep (evaluate.py)
        ↓
ConditionStep (RMSE ≤ rmse_threshold?)
    ├── TRUE  → ModelStep (crear) → TransformStep → ModelStep (registrar)
    └── FALSE → FailStep
```

## 1. Setup

In [ ]:
import sys
!{sys.executable} -m pip install "sagemaker>=2.99.0,<3.0" -q

In [ ]:
import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

# ── Sesión y credenciales ─────────────────────────────────────────────────────
sagemaker_session = sagemaker.session.Session()
pipeline_session  = PipelineSession()
region            = sagemaker_session.boto_region_name
role              = sagemaker.get_execution_role()

# ── Bucket y prefijos ─────────────────────────────────────────────────────────
bucket = "sagemaker-us-east-1-561137843164"
prefix = "ml-predsales-pipeline"

# ── Imágenes BYOC en ECR ──────────────────────────────────────────────────────
processing_image_uri = "561137843164.dkr.ecr.us-east-1.amazonaws.com/1c-preprocessing:latest"
training_image_uri   = "561137843164.dkr.ecr.us-east-1.amazonaws.com/ml-predsales-training:latest"

# ── Datos de entrada en S3 ────────────────────────────────────────────────────
input_data_uri = "s3://sagemaker-us-east-1-561137843164/sagemaker/1c-processing/input/raw/"

# ── Datos para batch transform (features del mes 34 generados por preprocessing) ─
# Se define después de correr el pipeline por primera vez;
# aquí se usa como placeholder que se actualizará con la URI real.
batch_data_uri = f"s3://{bucket}/{prefix}/batch-input/"

# ── Nombre del Model Package Group en el Model Registry ──────────────────────
model_package_group_name = "PredSalesModelPackageGroup"

print(f"Region:  {region}")
print(f"Role:    {role}")
print(f"Bucket:  {bucket}")
print(f"Prefix:  {prefix}")

## 2. Parámetros del pipeline

In [ ]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount", default_value=1
)
training_instance_type = ParameterString(
    name="TrainingInstanceType", default_value="ml.m5.large"
)
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
input_data = ParameterString(
    name="InputData", default_value=input_data_uri
)
batch_data = ParameterString(
    name="BatchData", default_value=batch_data_uri
)
# RMSE obtenido en tarea-05: 0.7442 — ponemos un umbral holgado para desarrollo
rmse_threshold = ParameterFloat(
    name="RmseThreshold", default_value=1.0
)

## 3. ProcessingStep — Preprocessing

Reutiliza el container BYOC `1c-preprocessing` de tarea-06.  
Genera tres outputs: `train` (meses 0–31), `validation` (mes 32) y `test` (mes 34, Kaggle).

In [ ]:
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.steps import ProcessingStep

script_processor = ScriptProcessor(
    image_uri=processing_image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=processing_instance_count,
    base_job_name="predsales-preprocessing",
    role=role,
    sagemaker_session=pipeline_session,
)

processor_args = script_processor.run(
    inputs=[
        ProcessingInput(
            source=input_data,
            destination="/opt/ml/processing/input",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="train",      source="/opt/ml/processing/output/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/output/validation"),
        ProcessingOutput(output_name="test",       source="/opt/ml/processing/output/test"),
    ],
    code="preprocess.py",
)

step_process = ProcessingStep(
    name="PredSalesPreprocess",
    step_args=processor_args,
)

## 4. TrainingStep

Reutiliza el container BYOC `ml-predsales-training` de tarea-05.  
Lee de los canales `train` y `validation` que genera el ProcessingStep anterior.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TrainingStep

model_output_path = f"s3://{bucket}/{prefix}/model"

estimator = Estimator(
    image_uri=training_image_uri,
    instance_type=training_instance_type,
    instance_count=1,
    output_path=model_output_path,
    base_job_name="predsales-training",
    role=role,
    sagemaker_session=pipeline_session,
)

train_args = estimator.fit(
    inputs={
        "training": TrainingInput(    # ← era "train"
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    }
)

step_train = TrainingStep(
    name="PredSalesTrain",
    step_args=train_args,
)

## 5. ProcessingStep — Evaluación

Reutiliza el mismo container BYOC de preprocessing para correr `evaluate.py`.  
Recibe el modelo del TrainingStep y los datos de validación del ProcessingStep anterior.  
Escribe `evaluation.json` con el RMSE que usará el ConditionStep.

In [ ]:
from sagemaker.workflow.properties import PropertyFile

script_eval = ScriptProcessor(
    image_uri=processing_image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="predsales-evaluation",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/input/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            destination="/opt/ml/processing/input/test",
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/output/evaluation",
        ),
    ],
    code="evaluate.py",
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

step_eval = ProcessingStep(
    name="PredSalesEval",
    step_args=eval_args,
    property_files=[evaluation_report],
)

## 6. ModelStep — Crear modelo

In [ ]:
from sagemaker.model import Model
from sagemaker.workflow.model_step import ModelStep

model = Model(
    image_uri=training_image_uri,
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
)

step_create_model = ModelStep(
    name="PredSalesCreateModel",
    step_args=model.create(instance_type="ml.m5.large"),
)

## 7. TransformStep — Batch transform

Corre inferencia batch sobre los features del mes 34 (test de Kaggle)  
generados por el ProcessingStep de preprocessing.

In [ ]:
from sagemaker.transformer import Transformer
from sagemaker.inputs import TransformInput
from sagemaker.workflow.steps import TransformStep

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/{prefix}/batch-output",
    accept="text/csv",
    assemble_with="Line",
    sagemaker_session=pipeline_session,
)

step_transform = TransformStep(
    name="PredSalesTransform",
    transformer=transformer,
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
        content_type="text/csv",
        split_type="Line",
    ),
)

## 8. ModelStep — Registrar modelo en el Model Registry

In [ ]:
from sagemaker.model_metrics import MetricsSource, ModelMetrics

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m5.large", "ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)

step_register = ModelStep(
    name="PredSalesRegisterModel",
    step_args=register_args,
)

## 9. FailStep

In [ ]:
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join

step_fail = FailStep(
    name="PredSalesRMSEFail",
    error_message=Join(
        on=" ",
        values=["Execution failed: RMSE superó el umbral de", rmse_threshold],
    ),
)

## 10. ConditionStep

Si `RMSE ≤ rmse_threshold` → registra el modelo y corre batch transform.  
Si no → ejecuta el FailStep.

In [ ]:
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet

cond_lte = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="regression_metrics.rmse.value",
    ),
    right=rmse_threshold,
)

step_cond = ConditionStep(
    name="PredSalesRMSECond",
    conditions=[cond_lte],
    if_steps=[step_create_model, step_transform, step_register],
    else_steps=[step_fail],
)

## 11. Definición y ejecución del pipeline

In [ ]:
from sagemaker.workflow.pipeline import Pipeline

pipeline_name = "PredSalesPipelineBYOC"

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        training_instance_type,
        model_approval_status,
        input_data,
        batch_data,
        rmse_threshold,
    ],
    steps=[step_process, step_train, step_eval, step_cond],
    sagemaker_session=pipeline_session,
)

In [ ]:
# Examina el JSON del pipeline antes de ejecutar (opcional pero útil para debug)
import json
definition = json.loads(pipeline.definition())
print(json.dumps(definition, indent=2, default=str))

In [ ]:
# Sube/actualiza el pipeline en SageMaker
pipeline.upsert(role_arn=role)
print(f"Pipeline '{pipeline_name}' registrado correctamente.")

In [ ]:
# Ejecuta el pipeline con los parámetros por defecto
execution = pipeline.start()
print(f"Execution ARN: {execution.arn}")

## 12. Verificación de resultados

In [ ]:
# Espera a que termine (puede tardar ~20-40 min)
try:
    execution.wait()
    print("Pipeline completado exitosamente.")
except Exception as e:
    print(f"Error en la ejecución: {e}")

In [ ]:
# Estado de cada step
execution.list_steps()

In [ ]:
# Lee el evaluation.json desde S3
import boto3
import json

s3_client = boto3.client("s3")

# Obtén la URI del output de evaluación
eval_output_uri = step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
eval_bucket = eval_output_uri.split("/")[2]
eval_key    = "/".join(eval_output_uri.split("/")[3:]) + "/evaluation.json"

response = s3_client.get_object(Bucket=eval_bucket, Key=eval_key)
report   = json.loads(response["Body"].read())

rmse_value = report["regression_metrics"]["rmse"]["value"]
print(f"RMSE del modelo: {rmse_value:.4f}")
print(f"Umbral:          {1.0}")
print(f"Modelo registrado: {'Sí' if rmse_value <= 1.0 else 'No'}")

In [ ]:
# Verifica artefactos en S3
paginator = s3_client.get_paginator("list_objects_v2")

print(f"=== Artefactos en s3://{bucket}/{prefix}/ ===")
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        print(" ", obj["Key"])

In [ ]:
# Verifica el modelo en el Model Registry
sm_client = boto3.client("sagemaker")

packages = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)["ModelPackageSummaryList"]

print(f"Modelos registrados en '{model_package_group_name}':")
for pkg in packages:
    print(f"  v{pkg['ModelPackageVersion']} | {pkg['ModelApprovalStatus']} | {pkg['CreationTime'].strftime('%Y-%m-%d %H:%M')}")

## Ejecución con umbral más estricto (opcional)

Para verificar que el FailStep funciona, ejecuta con un umbral imposible de cumplir:

In [ ]:
execution_fail = pipeline.start(
    parameters=dict(RmseThreshold=0.1)  # umbral imposible → debería ir al FailStep
)

try:
    execution_fail.wait()
except Exception as e:
    print(e)

execution_fail.list_steps()